# Contacts Data Cleaning

This notebook prepares the CRM **Contacts** table for downstream analysis.

### Main tasks
- standardize column names
- inspect data quality
- recover one invalid manager value using the raw Calls table
- identify and remove duplicate contacts
- create an ID replacement mapping for related tables
- convert date columns and validate chronological consistency

> **Data note:** the original CRM files are not included in the public repository.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Resolve project root whether the notebook is opened from the repository root
# or from the notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import to_snake, df_overview, df_clean_summary

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load and Initial Inspection

In [ ]:
contacts = pd.read_excel(RAW_DIR / 'Contacts (Done).xlsx', dtype={'Id': str})

In [ ]:
# Standardize column names to snake_case
contacts.columns = [to_snake(c) for c in contacts.columns]

In [ ]:
# Display a compact data-quality overview
df_overview(contacts)

In [ ]:
# Inspect unique values in non-ID columns
for col in contacts.columns:
    if col != 'id':
        print(f"\nUnique values in '{col}':")
        print(contacts[col].unique())

## 2. Recover an Invalid Manager Value

One contact contains `False` instead of a manager name.  
To avoid a circular dependency between the Contacts and cleaned Calls datasets, the manager is recovered directly from the **raw Calls table** using the contact ID.

In [ ]:
# Locate the contact with the invalid manager value
contacts[contacts['contact_owner_name'] == False]

In [ ]:
# Get the contact ID
false_id = contacts.loc[contacts['contact_owner_name'] == False, 'id'].iloc[0]
false_id

In [ ]:
# Load the raw Calls table only for manager recovery.
# This avoids depending on calls_clean.pkl before it has been created.
calls_raw = pd.read_excel(
    RAW_DIR / 'Calls (Done).xlsx',
    dtype={'Id': str, 'CONTACTID': str}
)
calls_raw.columns = [to_snake(c) for c in calls_raw.columns]
calls_raw.head(1)

In [ ]:
# Check which manager(s) handled calls associated with this contact
false_calls = calls_raw[calls_raw['contactid'] == false_id]
print(false_calls['call_owner_name'].value_counts())

In [ ]:
false_id_calls = calls_raw[calls_raw['contactid'] == false_id]
false_id_calls

In [ ]:
# Recover the manager name
false_id_manager = false_id_calls['call_owner_name']
false_id_manager.iloc[0]

In [ ]:
# Replace the invalid value with the recovered manager name
contacts.loc[contacts['contact_owner_name'] == False, 'contact_owner_name'] = false_id_manager.iloc[0]

In [ ]:
contacts[contacts['id'] == false_id]

## 3. Duplicate Contacts

Duplicate contacts are defined by matching manager, creation time, and modification time while having different IDs.

According to the project logic, these duplicates can occur when a manager creates a new CRM contact instead of reusing an existing one.

For each duplicate group:
- keep the record with the highest ID
- create an `old_id → main_id` mapping
- use this mapping later to reconnect Calls and Deals to the retained contact

In [ ]:
# Check exact duplicate rows
contacts.duplicated().sum()

In [ ]:
# Identify duplicates by manager and matching CRM timestamps
cols_dupl = ['contact_owner_name', 'created_time', 'modified_time']
contacts['id_num'] = pd.to_numeric(contacts['id'])  # compare IDs numerically

In [ ]:
is_dupl = contacts.duplicated(subset=cols_dupl, keep=False)  # mark all rows in duplicate groups

In [ ]:
contacts[is_dupl]

In [ ]:
# Group duplicate records and collect all IDs within each group
dupl_groups = (
    contacts[contacts.duplicated(subset=cols_dupl, keep=False)]
    .groupby(cols_dupl)['id_num']
    .agg(list)
    .reset_index()
)
dupl_groups

In [ ]:
# Build a replacement dictionary: old_id -> retained_id
id_replacements = {}
for ids in dupl_groups['id_num']:
    main = max(ids)
    for old_id in ids:
        if old_id != main:
            id_replacements[old_id] = main

In [ ]:
# Keep IDs as strings to match related CRM tables
id_replacements = {str(k): str(v) for k, v in id_replacements.items()}

In [ ]:
len_before = len(contacts)

contacts = (
    contacts
    .sort_values('id_num')
    .drop_duplicates(subset=cols_dupl, keep='last')
    .drop(columns=['id_num'])
    .reset_index(drop=True)
)

In [ ]:
print('Duplicates removed:', len_before - len(contacts))
print('Rows after deduplication:', contacts.shape[0])

## 4. Data Types

In [ ]:
# Parse CRM timestamps (DD.MM.YYYY HH:MM)
date_cols = ['created_time', 'modified_time']

for col in date_cols:
    contacts[col] = pd.to_datetime(
        contacts[col],
        format='%d.%m.%Y %H:%M',
        errors='coerce'
    )

In [ ]:
# Check whether parsing introduced missing datetime values
print(contacts[date_cols].isna().sum())

In [ ]:
print('Data types after conversion:')
print(contacts.dtypes)

## 5. Chronological Consistency Check

In [ ]:
(contacts['created_time'] > contacts['modified_time']).sum()

## 6. Final Quality Check

In [ ]:
df_clean_summary(contacts)

## 7. Save Processed Data

In [ ]:
# Pickle preserves parsed datetime and string types for downstream notebooks.
contacts.to_pickle(PROCESSED_DIR / 'contacts_clean.pkl')
pd.Series(id_replacements).to_pickle(
    PROCESSED_DIR / 'contacts_id_replacements.pkl'
)

print('Saved: contacts_clean.pkl')
print('Saved: contacts_id_replacements.pkl')
print(f'Shape: {contacts.shape}')

## 8. Output Dataset

| Column | Type | Description |
|---|---|---|
| `id` | `object` | Unique contact ID |
| `contact_owner_name` | `object` | Manager responsible for the contact |
| `created_time` | `datetime64` | Lead registration timestamp |
| `modified_time` | `datetime64` | Last CRM record modification timestamp |

**Dataset relationships**
- `contacts.id` ↔ `deals.contact_name`
- `contacts.id` ↔ `calls.contactid`

## 9. Cleaning Summary

**Source table:** CRM Contacts with four fields: `id`, `contact_owner_name`, `created_time`, `modified_time`.

| Step | Action | Result |
|---|---|---|
| 1 | Column standardization | Converted column names to `snake_case` |
| 2 | Invalid manager recovery | Recovered the manager name from the raw Calls table through the contact ID |
| 3 | Deduplication | Identified duplicates using manager + creation time + modification time; retained the highest ID in each group and created an ID replacement map |
| 4 | Datetime conversion | Converted `created_time` and `modified_time` to `datetime64` |
| 5 | Anomaly check | Validated that `created_time` does not occur after `modified_time` |

### Outputs

- `contacts_clean.pkl` — cleaned Contacts dataset
- `contacts_id_replacements.pkl` — `old_id → retained_id` mapping for reconnecting related Calls and Deals records